# Laya GPU test on Kaggle T4x2

Keep this notebook private. Enable GPU T4x2 and Internet. No dataset upload is needed. The runner downloads only allowlisted files from the published test checkpoint commit, verifies model hashes, and generates the approved-pilot replay plus synthetic cases. The gold split is excluded.

The run uses one GPU with micro-batch 16 and an effective batch of 16. A 64-row smoke checks CUDA and Laya compatibility before two more fine-tuning passes. Outputs stay under `/kaggle/working/laya-pilot`.


In [ ]:
from pathlib import Path
import base64, hashlib, json, subprocess, sys
from urllib.request import Request, urlopen

RUNNER = Path('/kaggle/working/sid_kaggle.py')
INPUT = Path('/kaggle/working/laya-inputs')
OUTPUT = Path('/kaggle/working/laya-pilot')
request = Request('https://api.github.com/repos/trigtbh/dubhacks-26/git/blobs/e15ea8cc6332aea8d4468295e63039503c189d2e',
                  headers={'User-Agent': 'sid-laya-kaggle'})
with urlopen(request, timeout=60) as response:
    source = base64.b64decode(json.load(response)['content'])
if hashlib.sha256(source).hexdigest() != '06aa112d8bfe615f55990543292b5bb5d9ab63fa5ef1d56f3ba0fd5c70f28185':
    raise RuntimeError('Kaggle runner checksum mismatch')
RUNNER.write_bytes(source)
print('Verified runner:', RUNNER)


## Install the pinned Python packages

First record the already installed CUDA Torch version. The install cell pins Laya, Transformers, and their non-Torch dependencies while constraining Torch to its current version, so pip cannot silently replace the Kaggle CUDA build. The model-load and training smoke then check actual compatibility.

In [ ]:
import importlib.metadata, torch
if not torch.cuda.is_available() or torch.version.hip is not None:
    raise RuntimeError('Enable Kaggle GPU and use a CUDA PyTorch build before installing packages')
torch_before = torch.__version__
constraint = Path('/kaggle/working/torch-constraint.txt')
constraint.write_text(f'torch=={torch_before}\n')
subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-c', str(constraint),
                      'laya==0.4.11', 'transformers==5.20.1', 'tokenizers==0.23.3',
                      'huggingface-hub>=1.31,<3', 'safetensors>=0.8', 'regex>=2025.10.22'])
import torch as torch_after_check
if torch_after_check.__version__ != torch_before:
    raise RuntimeError(f'Package install changed Torch from {torch_before} to {torch_after_check.__version__}')
print('Torch preserved:', torch_before)

## Download and verify the pinned inputs

This resumes the published experimental test candidate. The initial weights include earlier experimental training. Demo cases share command families with training, so their scores measure known regressions. CUDA control scores are recomputed from the original checkpoint.


In [ ]:
subprocess.run([sys.executable, '-B', '-u', str(RUNNER), 'prepare', '--input', str(INPUT)], check=True)
subprocess.run([sys.executable, '-B', '-u', str(RUNNER), 'verify', '--input', str(INPUT)], check=True)


## Check CUDA training on 64 rows

The elapsed smoke time includes model loading and checkpoint writes. It is a compatibility check, not an epoch-throughput estimate. If batch 16 exhausts memory, set `--micro-batch` to `8` or `4` in both training cells.


In [ ]:
subprocess.run([sys.executable, '-B', '-u', str(RUNNER), 'smoke', '--input', str(INPUT),
                '--output', str(OUTPUT), '--micro-batch', '16'], check=True)


## Continue fine-tuning and check candidates

Two additional passes update the last two encoder layers and classification head. Selection requires all 768 known-command contexts to pass twice, no increase in incorrect Allow on the control or prior Block checks, and control accuracy within two percentage points of the fresh CUDA baseline.


In [ ]:
subprocess.run([sys.executable, '-B', '-u', str(RUNNER), 'train', '--input', str(INPUT),
                '--output', str(OUTPUT), '--micro-batch', '16'], check=True)


In [ ]:
for path in sorted(OUTPUT.rglob('*')):
    if path.is_file():
        print(path.relative_to(OUTPUT), path.stat().st_size)
selection = OUTPUT / 'selection.json'
if selection.exists():
    print(json.dumps(json.loads(selection.read_text()), indent=2))